# Biologically Realistic Cochlear Model (Nengo / NEF)

A spiking-neuron model of cochlear frequency analysis, built with Nengo's
Neural Engineering Framework. Each ensemble of LIF neurons represents the
activity of auditory-nerve fibers tuned to one cochlear place (frequency
channel), reproducing the tonotopic organization of the real cochlea.

## Files

- **`cochlear_filterbank.py`** — Mechanical stage. Implements:
  - The **Greenwood (1990) function**, the standard map from physical
    position along the basilar membrane to characteristic frequency (CF).
  - The **ERB scale** (Glasberg & Moore, 1990) for realistic, frequency-
    dependent auditory filter bandwidths (filters widen at high CF).
  - A **gammatone filterbank** (Slaney 1993 implementation) — the standard
    signal-processing proxy for basilar-membrane traveling-wave mechanics.
    Channels are spaced evenly in cochlear *position*, not in Hz, so they
    naturally bunch up at low frequencies exactly as in a real cochlea.

- **`cochlear_nef_model.py`** — Neural stage. Implements:
  - **Inner hair cell transduction**: half-wave rectification (hair cells
    only respond to one direction of membrane motion) and power-law
    compression (the cochlear amplifier's compressive input/output
    function, the basis of the ear's ~120 dB dynamic range).
  - **One independent NEF ensemble per channel**: LIF neurons with
    positive-only encoders (matching the fact that hair-cell-to-nerve
    drive is never negative) and a ~0.75 ms absolute refractory period
    (matching the standard Zilany/Bruce/Carney auditory-periphery model;
    see Bruce, Erfani & Zilany, *Hearing Research* 360, 2018). Each channel's representational radius is
    scaled to its own signal range so weak high-frequency channels aren't
    swamped by louder low-frequency ones.
  - **Use two separate models for binaural (stereo) inputs.** 

- **`demo_visualization.py`** — Drives the model with a logarithmic chirp
  (100→20000 Hz) followed by a three-tone chord, and produces:
  1. The stimulus waveform
  2. A spike raster, channels ordered low-CF → high-CF
  3. A **cochleagram** (smoothed per-channel firing rate over time) —
     this is the clearest view of tonotopy: the chirp produces a clean
     diagonal traveling-wave band sweeping from low to high channels, and
     the chord lights up three separate horizontal bands simultaneously.

## Usage

```python
from cochlear_nef_model import CochlearModel
import nengo
import numpy as np

fs = 20000
model = CochlearModel(fs=fs, n_channels=32, low_freq=2000, high_freq=60000,
                       neurons_per_channel=25)

t = np.arange(0, 0.5, 1/fs)
audio = np.sin(2 * np.pi * 1000 * t)  # your audio signal here
model.set_audio(audio)

net, probes = model.build()
with nengo.Simulator(net) as sim:
    sim.run(0.5)

# probes["spikes"][i]  -> spike probe for channel i (CF = model.cfs[i])
# probes["envelope"][i]-> decoded (synaptically filtered) firing rate for channel i
```
### Potential dataset for validating the model
https://doi.org/10.5061/dryad.qv9s4mwn4
